In [22]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nguyenphantuanduy/art-musique-code/musique_teacher_prompt.py
/kaggle/input/datasets/nguyenphantuanduy/art-musique-code/musique_sample.py
/kaggle/input/datasets/nguyenphantuanduy/art-musique-code/load_dataset.py
/kaggle/input/datasets/nguyenphantuanduy/art-musique-code/dataset.py


In [23]:
# ============================================================
# RESET GPU MEMORY
# ============================================================

import gc
import torch

print("=" * 80)
print("RESET GPU MEMORY")
print("=" * 80)

# Delete common objects if they exist.
for name in [
    "outputs",
    "inputs",
    "model",
    "tokenizer",
]:
    if name in globals():
        del globals()[name]

# Python garbage collection.
gc.collect()

# Clear PyTorch CUDA cache.
if torch.cuda.is_available():

    torch.cuda.empty_cache()

    # Synchronize GPU.
    torch.cuda.synchronize()

    allocated = (
        torch.cuda.memory_allocated()
        / 1024**3
    )

    reserved = (
        torch.cuda.memory_reserved()
        / 1024**3
    )

    print(
        f"Allocated VRAM : {allocated:.3f} GB"
    )

    print(
        f"Reserved VRAM  : {reserved:.3f} GB"
    )

print()
print("✓ GPU memory cleanup completed")

RESET GPU MEMORY
Allocated VRAM : 0.008 GB
Reserved VRAM  : 0.021 GB

✓ GPU memory cleanup completed


In [24]:
import sys
from pathlib import Path

CODE_DIR = Path(
    "/kaggle/input/datasets/nguyenphantuanduy/art-musique-code"
)

sys.path.insert(0, str(CODE_DIR))

print("✓ Code directory:")
print(CODE_DIR)

✓ Code directory:
/kaggle/input/datasets/nguyenphantuanduy/art-musique-code


In [25]:
from musique_sample import (
    MuSiQueSampleConverter,
)

from musique_teacher_prompt import (
    create_teacher_prompt,
)

from load_dataset import (
    load_musique_ans,
)

print("✓ All imports successful")

✓ All imports successful


In [26]:
import gdown
import zipfile
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================

MUSIQUE_FILE_ID = "1tGdADlNjWFaHLeZZGShh2IRcpO6Lv24h"

WORK_DIR = Path("/kaggle/working/musique")
ZIP_PATH = WORK_DIR / "musique_v1.0.zip"
DATA_DIR = WORK_DIR / "data"

WORK_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ============================================================
# DOWNLOAD
# ============================================================

if not ZIP_PATH.exists():

    print("=" * 80)
    print("DOWNLOADING MuSiQue v1.0")
    print("=" * 80)

    gdown.download(
        id=MUSIQUE_FILE_ID,
        output=str(ZIP_PATH),
        quiet=False,
    )

else:

    print("✓ MuSiQue ZIP already exists")

# ============================================================
# EXTRACT
# ============================================================

if not (DATA_DIR / "musique_ans_v1.0_dev.jsonl").exists():

    print()
    print("=" * 80)
    print("EXTRACTING MuSiQue v1.0")
    print("=" * 80)

    with zipfile.ZipFile(
        ZIP_PATH,
        "r",
    ) as z:

        z.extractall(
            WORK_DIR
        )

else:

    print("✓ MuSiQue already extracted")

print()
print("Data directory:")
print(DATA_DIR)

✓ MuSiQue ZIP already exists
✓ MuSiQue already extracted

Data directory:
/kaggle/working/musique/data


In [27]:
import json

def load_jsonl(path):
    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:

        return [
            json.loads(line)
            for line in f
        ]


# ============================================================
# LOAD DEV
# ============================================================

dev_path = (
    DATA_DIR
    / "musique_ans_v1.0_dev.jsonl"
)

dev_dataset = load_jsonl(
    dev_path
)

print("=" * 80)
print("MuSiQue DEV LOADED")
print("=" * 80)

print(
    f"Samples: {len(dev_dataset):,}"
)

MuSiQue DEV LOADED
Samples: 2,417


In [28]:
raw_sample = dev_dataset[0]

print("=" * 80)
print("RAW SAMPLE — DEV[0]")
print("=" * 80)

print(
    "Question:",
    raw_sample["question"],
)

print(
    "Answer:",
    raw_sample["answer"],
)

print(
    "Paragraphs:",
    len(raw_sample["paragraphs"]),
)

print(
    "Reasoning steps:",
    len(raw_sample["question_decomposition"]),
)

RAW SAMPLE — DEV[0]
Question: Who is the spouse of the Green performer?
Answer: Miquette Giraudy
Paragraphs: 20
Reasoning steps: 2


In [29]:
sample = (
    MuSiQueSampleConverter.from_musique(
        raw_sample
    )
)

print("=" * 80)
print("MUSIQUE SAMPLE CONVERTED")
print("=" * 80)

print(
    f"Question             : {sample.question}"
)

print(
    f"Answer               : {sample.answer}"
)

print(
    f"Golden documents     : "
    f"{len(sample.golden_document)}"
)

print(
    f"Distractor documents : "
    f"{len(sample.distract_document)}"
)

print(
    f"Reasoning steps      : "
    f"{len(sample.question_decomposition)}"
)

print(
    f"Reasoning instruction: "
    f"{len(sample.reasoning_instruction)} chars"
)

print(
    "\n✓ Conversion successful"
)

MUSIQUE SAMPLE CONVERTED
Question             : Who is the spouse of the Green performer?
Answer               : Miquette Giraudy
Golden documents     : 2
Distractor documents : 18
Reasoning steps      : 2
Reasoning instruction: 562 chars

✓ Conversion successful


In [30]:
teacher_prompt = create_teacher_prompt(
    sample=sample
)

print("=" * 80)
print("GENERATED TEACHER PROMPT")
print("=" * 80)

print(teacher_prompt)

GENERATED TEACHER PROMPT
You are a reasoning teacher generating high-quality reasoning
data for a retrieval-augmented language model.

Your task is to solve the question below by reasoning over the
provided documents.

The target answer is provided intentionally as supervision for
this teacher-generation process.

IMPORTANT:

- You are given the final answer as a target.
- Reconstruct a natural reasoning process that leads to that
  answer.
- Do NOT simply explain the answer directly.
- Start from the original question.
- Determine what information is required.
- Identify the relevant evidence.
- Connect intermediate facts when the question requires
  multi-hop reasoning.
- Every factual claim should be grounded in the provided
  documents whenever possible.
- When referring to a document, ALWAYS use its exact Title.
- NEVER refer to documents as "Document 1", "Document 2", etc.
- Do not mention that you were given the target answer.
- Do not mention internal dataset annotations.
- Do 

In [31]:
# ============================================================
# LOAD QWEN3-4B TEACHER
# ============================================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-4B"

print("=" * 80)
print("LOADING TEACHER MODEL")
print("=" * 80)

print(f"Model: {MODEL_NAME}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(
        f"Total VRAM: "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

model.eval()

print()
print("✓ Tokenizer loaded")
print("✓ Model loaded")
print(
    f"Model device: {model.device}"
)

if torch.cuda.is_available():
    print(
        f"Allocated VRAM: "
        f"{torch.cuda.memory_allocated() / 1024**3:.3f} GB"
    )

    print(
        f"Reserved VRAM: "
        f"{torch.cuda.memory_reserved() / 1024**3:.3f} GB"
    )

LOADING TEACHER MODEL
Model: Qwen/Qwen3-4B
CUDA available: True
GPU: Tesla T4
Total VRAM: 14.56 GB


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


✓ Tokenizer loaded
✓ Model loaded
Model device: cuda:0
Allocated VRAM: 3.740 GB
Reserved VRAM: 3.762 GB


In [32]:
# ============================================================
# BUILD CHAT INPUT
# ============================================================

messages = [
    {
        "role": "user",
        "content": teacher_prompt,
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

inputs = tokenizer(
    text,
    return_tensors="pt",
)

# Move inputs to the same device as the model.
inputs = {
    key: value.to(model.device)
    for key, value in inputs.items()
}

print("=" * 80)
print("CHAT TEMPLATE")
print("=" * 80)

print(
    f"Input tokens: "
    f"{inputs['input_ids'].shape[-1]}"
)

print(
    f"Input shape: "
    f"{inputs['input_ids'].shape}"
)

print()
print("✓ Chat template applied")
print("✓ Input tokenized")

CHAT TEMPLATE
Input tokens: 883
Input shape: torch.Size([1, 883])

✓ Chat template applied
✓ Input tokenized


In [33]:
# ============================================================
# GENERATE TEACHER REASONING
# MULTI-GPU VRAM PROFILING
# ============================================================

import time
import torch

print("=" * 80)
print("TEACHER GENERATION")
print("MULTI-GPU VRAM PROFILING")
print("=" * 80)

# ============================================================
# GPU INFORMATION
# ============================================================

num_gpus = torch.cuda.device_count()

print()
print(f"Number of GPUs: {num_gpus}")

for i in range(num_gpus):

    total_vram = (
        torch.cuda.get_device_properties(i).total_memory
        / 1024**3
    )

    print(
        f"GPU {i}: "
        f"{torch.cuda.get_device_name(i)} "
        f"({total_vram:.3f} GB)"
    )

# ============================================================
# RESET PEAK MEMORY STATISTICS
# ============================================================

for i in range(num_gpus):

    with torch.cuda.device(i):

        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

# ============================================================
# INITIAL VRAM
# ============================================================

initial_allocated = []
initial_reserved = []

print()
print("=" * 80)
print("INITIAL VRAM")
print("=" * 80)

for i in range(num_gpus):

    allocated = (
        torch.cuda.memory_allocated(i)
        / 1024**3
    )

    reserved = (
        torch.cuda.memory_reserved(i)
        / 1024**3
    )

    initial_allocated.append(allocated)
    initial_reserved.append(reserved)

    print(
        f"GPU {i} | "
        f"Allocated: {allocated:.3f} GB | "
        f"Reserved: {reserved:.3f} GB"
    )

print(
    f"TOTAL    | "
    f"Allocated: {sum(initial_allocated):.3f} GB | "
    f"Reserved: {sum(initial_reserved):.3f} GB"
)

# ============================================================
# GENERATION
# ============================================================

start_time = time.perf_counter()

with torch.inference_mode():

    outputs = model.generate(
        **inputs,

        max_new_tokens=1024,

        do_sample=True,
        temperature=0.6,
        top_p=0.95,

        pad_token_id=tokenizer.eos_token_id,
    )

generation_time = (
    time.perf_counter()
    - start_time
)

# ============================================================
# SYNCHRONIZE ALL GPUs
# ============================================================

for i in range(num_gpus):

    torch.cuda.synchronize(i)

# ============================================================
# PEAK VRAM
# ============================================================

peak_allocated = []
peak_reserved = []

print()
print("=" * 80)
print("PEAK VRAM")
print("=" * 80)

for i in range(num_gpus):

    allocated = (
        torch.cuda.max_memory_allocated(i)
        / 1024**3
    )

    reserved = (
        torch.cuda.max_memory_reserved(i)
        / 1024**3
    )

    peak_allocated.append(allocated)
    peak_reserved.append(reserved)

    print(
        f"GPU {i} | "
        f"Peak allocated: {allocated:.3f} GB | "
        f"Peak reserved: {reserved:.3f} GB"
    )

total_peak_allocated = sum(peak_allocated)
total_peak_reserved = sum(peak_reserved)

print(
    f"TOTAL    | "
    f"Peak allocated: {total_peak_allocated:.3f} GB | "
    f"Peak reserved: {total_peak_reserved:.3f} GB"
)

# ============================================================
# DECODE ONLY GENERATED TOKENS
# ============================================================

input_length = inputs["input_ids"].shape[-1]

generated_tokens = outputs[
    0,
    input_length:
]

generated_text = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True,
)

# ============================================================
# OUTPUT
# ============================================================

print()
print("=" * 80)
print("GENERATED REASONING")
print("=" * 80)

print(generated_text)

# ============================================================
# GENERATION STATISTICS
# ============================================================

print()
print("=" * 80)
print("GENERATION STATISTICS")
print("=" * 80)

print(
    f"Input tokens        : "
    f"{input_length}"
)

print(
    f"Generated tokens     : "
    f"{len(generated_tokens)}"
)

print(
    f"Generation time      : "
    f"{generation_time:.2f} sec"
)

if generation_time > 0:

    print(
        f"Generation speed    : "
        f"{len(generated_tokens) / generation_time:.2f} tokens/sec"
    )

print()
print(
    f"Total peak allocated : "
    f"{total_peak_allocated:.3f} GB"
)

print(
    f"Total peak reserved  : "
    f"{total_peak_reserved:.3f} GB"
)

print()
print("=" * 80)
print("PER-GPU PEAK SUMMARY")
print("=" * 80)

for i in range(num_gpus):

    print(
        f"GPU {i}: "
        f"{peak_allocated[i]:.3f} GB allocated / "
        f"{peak_reserved[i]:.3f} GB reserved"
    )

TEACHER GENERATION
MULTI-GPU VRAM PROFILING

Number of GPUs: 2
GPU 0: Tesla T4 (14.562 GB)
GPU 1: Tesla T4 (14.562 GB)

INITIAL VRAM
GPU 0 | Allocated: 3.740 GB | Reserved: 3.762 GB
GPU 1 | Allocated: 3.768 GB | Reserved: 3.789 GB
TOTAL    | Allocated: 7.508 GB | Reserved: 7.551 GB

PEAK VRAM
GPU 0 | Peak allocated: 4.094 GB | Peak reserved: 4.139 GB
GPU 1 | Peak allocated: 4.138 GB | Peak reserved: 4.188 GB
TOTAL    | Peak allocated: 8.232 GB | Peak reserved: 8.326 GB

GENERATED REASONING
The question asks for the spouse of the Green performer. To answer this, we first need to identify who the performer associated with the album "Green" is. The document "Green (Steve Hillage album)" states that "Green" is the fourth studio album by British progressive rock musician Steve Hillage. This establishes that the performer linked to "Green" is Steve Hillage.  

Next, we need to determine who Steve Hillage's spouse is. The document "Miquette Giraudy" mentions that Miquette Giraudy is best know

In [43]:
# ============================================================
# CREATE 16 TEACHER PROMPTS
# ============================================================

BATCH_SIZE = 16

print("=" * 80)
print("CREATING 16 TEACHER PROMPTS")
print("=" * 80)

# ------------------------------------------------------------
# Create prompts directly from existing dev_dataset
# ------------------------------------------------------------

teacher_prompts = []

for i in range(BATCH_SIZE):

    # --------------------------------------------------------
    # Load raw sample
    # --------------------------------------------------------

    raw_sample = dev_dataset[i]

    # --------------------------------------------------------
    # Convert MuSiQue sample
    # --------------------------------------------------------

    sample = (
        MuSiQueSampleConverter.from_musique(
            raw_sample
        )
    )

    # --------------------------------------------------------
    # Create teacher prompt
    # --------------------------------------------------------

    teacher_prompt = create_teacher_prompt(
        sample=sample
    )

    teacher_prompts.append(
        teacher_prompt
    )

# ============================================================
# VALIDATION
# ============================================================

assert len(teacher_prompts) == BATCH_SIZE

assert all(
    isinstance(prompt, str)
    and len(prompt) > 0
    for prompt in teacher_prompts
)

# ============================================================
# SUMMARY
# ============================================================

print()
print("=" * 80)
print("TEACHER PROMPT BATCH")
print("=" * 80)

print(
    f"Number of prompts : "
    f"{len(teacher_prompts)}"
)

print(
    f"Total characters  : "
    f"{sum(len(p) for p in teacher_prompts):,}"
)

print(
    f"Average characters: "
    f"{sum(len(p) for p in teacher_prompts) / len(teacher_prompts):,.0f}"
)

print(
    f"Min characters    : "
    f"{min(len(p) for p in teacher_prompts):,}"
)

print(
    f"Max characters    : "
    f"{max(len(p) for p in teacher_prompts):,}"
)

print()
print("✓ 16 teacher prompts created successfully")

CREATING 16 TEACHER PROMPTS

TEACHER PROMPT BATCH
Number of prompts : 16
Total characters  : 71,877
Average characters: 4,492
Min characters    : 4,168
Max characters    : 4,976

✓ 16 teacher prompts created successfully


In [44]:
# ============================================================
# BUILD BATCH CHAT INPUT
# ============================================================

print("=" * 80)
print("BUILD BATCH CHAT INPUT")
print("=" * 80)

messages_batch = [
    [
        {
            "role": "user",
            "content": teacher_prompt,
        }
    ]
    for teacher_prompt in teacher_prompts
]

# ------------------------------------------------------------
# Apply chat template
# ------------------------------------------------------------

texts = [
    tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    for messages in messages_batch
]

# ------------------------------------------------------------
# Tokenize as batch
# ------------------------------------------------------------

inputs = tokenizer(
    texts,
    return_tensors="pt",
    padding=True,
    truncation=False,
)

# ------------------------------------------------------------
# Move inputs to the same device as the model
# ------------------------------------------------------------

inputs = {
    key: value.to(model.device)
    for key, value in inputs.items()
}

# ============================================================
# STATISTICS
# ============================================================

input_lengths = (
    inputs["attention_mask"]
    .sum(dim=1)
    .tolist()
)

print()
print("=" * 80)
print("BATCH INPUT STATISTICS")
print("=" * 80)

print(
    f"Batch size               : "
    f"{len(texts)}"
)

print(
    f"Batch tensor shape       : "
    f"{inputs['input_ids'].shape}"
)

print(
    f"Max input tokens         : "
    f"{max(input_lengths)}"
)

print(
    f"Min input tokens         : "
    f"{min(input_lengths)}"
)

print(
    f"Average input tokens     : "
    f"{sum(input_lengths) / len(input_lengths):.1f}"
)

print(
    f"Total actual input tokens: "
    f"{sum(input_lengths)}"
)

print()
print("✓ 16 prompts converted to chat format")
print("✓ Batch tokenization completed")

BUILD BATCH CHAT INPUT

BATCH INPUT STATISTICS
Batch size               : 16
Batch tensor shape       : torch.Size([16, 985])
Max input tokens         : 985
Min input tokens         : 742
Average input tokens     : 853.2
Total actual input tokens: 13652

✓ 16 prompts converted to chat format
✓ Batch tokenization completed


In [45]:
# ============================================================
# CLEAN GPU MEMORY AFTER OOM
# ============================================================

import gc
import torch

print("=" * 80)
print("CLEANING GPU MEMORY")
print("=" * 80)

# ------------------------------------------------------------
# Delete temporary generation / input tensors
# ------------------------------------------------------------

for name in [
    "outputs",
    "generated_tokens",
    "generated_text",
    "inputs",
    "texts",
    "messages_batch",
]:

    if name in globals():

        del globals()[name]

# ------------------------------------------------------------
# Python garbage collection
# ------------------------------------------------------------

gc.collect()

# ------------------------------------------------------------
# Clear CUDA cache and reset peak statistics
# ------------------------------------------------------------

num_gpus = torch.cuda.device_count()

for i in range(num_gpus):

    with torch.cuda.device(i):

        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()

        torch.cuda.synchronize()

# ============================================================
# MEMORY STATUS
# ============================================================

print()
print("=" * 80)
print("GPU MEMORY AFTER CLEANUP")
print("=" * 80)

total_allocated = 0.0
total_reserved = 0.0

for i in range(num_gpus):

    allocated = (
        torch.cuda.memory_allocated(i)
        / 1024**3
    )

    reserved = (
        torch.cuda.memory_reserved(i)
        / 1024**3
    )

    total_allocated += allocated
    total_reserved += reserved

    print(
        f"GPU {i} | "
        f"Allocated: {allocated:.3f} GB | "
        f"Reserved: {reserved:.3f} GB"
    )

print()
print(
    f"TOTAL | "
    f"Allocated: {total_allocated:.3f} GB | "
    f"Reserved: {total_reserved:.3f} GB"
)

print()
print("✓ Python garbage collection completed")
print("✓ CUDA cache cleared")
print("✓ Peak memory statistics reset")
print("✓ GPU memory ready for next test")

CLEANING GPU MEMORY

GPU MEMORY AFTER CLEANUP
GPU 0 | Allocated: 4.982 GB | Reserved: 5.609 GB
GPU 1 | Allocated: 3.784 GB | Reserved: 3.807 GB

TOTAL | Allocated: 8.766 GB | Reserved: 9.416 GB

✓ Python garbage collection completed
✓ CUDA cache cleared
✓ Peak memory statistics reset
✓ GPU memory ready for next test


In [49]:
# ============================================================
# BATCH 16 TEACHER GENERATION
# LEFT PADDING + GENERATION + VRAM PROFILING
# ============================================================

import gc
import time
import torch

# ============================================================
# CONFIG
# ============================================================

BATCH_SIZE = 16
MAX_NEW_TOKENS = 1024

print("=" * 80)
print("BATCH 16 TEACHER GENERATION")
print("LEFT PADDING + GENERATION + VRAM PROFILING")
print("=" * 80)

# ============================================================
# 1. CLEAN PYTHON / CUDA CACHE
# ============================================================

gc.collect()

if torch.cuda.is_available():

    for i in range(torch.cuda.device_count()):

        with torch.cuda.device(i):

            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.reset_accumulated_memory_stats()

# ============================================================
# 2. CREATE 16 TEACHER PROMPTS
# ============================================================

teacher_prompts = []

for i in range(BATCH_SIZE):

    raw_sample = dev_dataset[i]

    sample = (
        MuSiQueSampleConverter.from_musique(
            raw_sample
        )
    )

    teacher_prompt = create_teacher_prompt(
        sample=sample
    )

    teacher_prompts.append(
        teacher_prompt
    )

assert len(teacher_prompts) == BATCH_SIZE

print()
print(
    f"✓ Created {len(teacher_prompts)} teacher prompts"
)

# ============================================================
# 3. APPLY CHAT TEMPLATE
# ============================================================

messages_batch = [
    [
        {
            "role": "user",
            "content": teacher_prompt,
        }
    ]
    for teacher_prompt in teacher_prompts
]

texts = [
    tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    for messages in messages_batch
]

# ============================================================
# 4. TOKENIZER — LEFT PADDING
# ============================================================

# Save original padding side so it can be restored later.
original_padding_side = tokenizer.padding_side

tokenizer.padding_side = "left"

inputs = tokenizer(
    texts,
    return_tensors="pt",
    padding=True,
    truncation=False,
)

# Restore original tokenizer setting.
tokenizer.padding_side = original_padding_side

# ------------------------------------------------------------
# Move inputs to model input device
# ------------------------------------------------------------

inputs = {
    key: value.to(model.device)
    for key, value in inputs.items()
}

# ============================================================
# 5. INPUT STATISTICS
# ============================================================

input_lengths = (
    inputs["attention_mask"]
    .sum(dim=1)
    .tolist()
)

padded_input_length = (
    inputs["input_ids"].shape[1]
)

total_actual_input_tokens = sum(
    input_lengths
)

print()
print("=" * 80)
print("BATCH INPUT")
print("=" * 80)

print(
    f"Batch size               : "
    f"{BATCH_SIZE}"
)

print(
    f"Padding side             : "
    f"left"
)

print(
    f"Batch tensor shape       : "
    f"{tuple(inputs['input_ids'].shape)}"
)

print(
    f"Padded input length      : "
    f"{padded_input_length}"
)

print(
    f"Min actual input tokens  : "
    f"{min(input_lengths)}"
)

print(
    f"Max actual input tokens  : "
    f"{max(input_lengths)}"
)

print(
    f"Average actual tokens    : "
    f"{total_actual_input_tokens / BATCH_SIZE:.1f}"
)

print(
    f"Total actual input tokens: "
    f"{total_actual_input_tokens}"
)

# ============================================================
# 6. GPU INFORMATION
# ============================================================

num_gpus = torch.cuda.device_count()

print()
print("=" * 80)
print("GPU INFORMATION")
print("=" * 80)

print(
    f"Number of GPUs: {num_gpus}"
)

for i in range(num_gpus):

    total_vram = (
        torch.cuda.get_device_properties(i)
        .total_memory
        / 1024**3
    )

    print(
        f"GPU {i}: "
        f"{torch.cuda.get_device_name(i)} "
        f"({total_vram:.3f} GB)"
    )

# ============================================================
# 7. INITIAL VRAM
# ============================================================

initial_allocated = []
initial_reserved = []

print()
print("=" * 80)
print("INITIAL VRAM")
print("=" * 80)

for i in range(num_gpus):

    allocated = (
        torch.cuda.memory_allocated(i)
        / 1024**3
    )

    reserved = (
        torch.cuda.memory_reserved(i)
        / 1024**3
    )

    initial_allocated.append(
        allocated
    )

    initial_reserved.append(
        reserved
    )

    print(
        f"GPU {i} | "
        f"Allocated: {allocated:.3f} GB | "
        f"Reserved: {reserved:.3f} GB"
    )

print(
    f"TOTAL    | "
    f"Allocated: {sum(initial_allocated):.3f} GB | "
    f"Reserved: {sum(initial_reserved):.3f} GB"
)

# ============================================================
# 8. GENERATION
# ============================================================

print()
print("=" * 80)
print("GENERATING")
print("=" * 80)

start_time = time.perf_counter()

generation_success = False
outputs = None

try:

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,

            max_new_tokens=MAX_NEW_TOKENS,

            do_sample=True,
            temperature=0.6,
            top_p=0.95,

            pad_token_id=tokenizer.eos_token_id,
        )

    generation_success = True

except torch.cuda.OutOfMemoryError as e:

    print()
    print("=" * 80)
    print("CUDA OUT OF MEMORY")
    print("=" * 80)

    print(e)

# ------------------------------------------------------------
# Synchronize all GPUs
# ------------------------------------------------------------

for i in range(num_gpus):

    torch.cuda.synchronize(i)

generation_time = (
    time.perf_counter()
    - start_time
)

# ============================================================
# 9. PEAK VRAM
# ============================================================

peak_allocated = []
peak_reserved = []

print()
print("=" * 80)
print("PEAK VRAM")
print("=" * 80)

for i in range(num_gpus):

    allocated = (
        torch.cuda.max_memory_allocated(i)
        / 1024**3
    )

    reserved = (
        torch.cuda.max_memory_reserved(i)
        / 1024**3
    )

    peak_allocated.append(
        allocated
    )

    peak_reserved.append(
        reserved
    )

    print(
        f"GPU {i} | "
        f"Peak allocated: {allocated:.3f} GB | "
        f"Peak reserved: {reserved:.3f} GB"
    )

total_peak_allocated = sum(
    peak_allocated
)

total_peak_reserved = sum(
    peak_reserved
)

print()
print(
    f"TOTAL | Peak allocated: "
    f"{total_peak_allocated:.3f} GB"
)

print(
    f"TOTAL | Peak reserved : "
    f"{total_peak_reserved:.3f} GB"
)

# ============================================================
# 10. GENERATION FAILED
# ============================================================

if not generation_success:

    print()
    print("=" * 80)
    print("GENERATION FAILED")
    print("=" * 80)

    print(
        "Batch generation caused CUDA OOM."
    )

    print(
        "Peak VRAM above shows the memory reached before OOM."
    )

# ============================================================
# 11. PROCESS SUCCESSFUL GENERATION
# ============================================================

else:

    # ========================================================
    # IMPORTANT:
    #
    # With LEFT PADDING:
    #
    # inputs:
    #
    # [PAD PAD token token token]
    # [PAD token token token token]
    #
    # All generated tokens are appended AFTER the padded
    # input width.
    #
    # Therefore:
    #
    # outputs[:, padded_input_length:]
    #
    # is the correct generated region.
    # ========================================================

    output_sequence_length = (
        outputs.shape[1]
    )

    generated_tokens_batch = outputs[
        :,
        padded_input_length:
    ]

    # ========================================================
    # 12. GENERATED LENGTHS
    # ========================================================

    # All samples share the same generated region because
    # generation starts after the padded input width.
    #
    # However, some samples may stop early and therefore contain
    # EOS/PAD tokens. We count non-padding generated tokens.

    generated_lengths = []

    for i in range(BATCH_SIZE):

        generated_tokens = (
            generated_tokens_batch[i]
        )

        # Count tokens before EOS.
        eos_positions = (
            generated_tokens
            == tokenizer.eos_token_id
        )

        eos_indices = torch.nonzero(
            eos_positions,
            as_tuple=False
        )

        if eos_indices.numel() > 0:

            generated_len = int(
                eos_indices[0].item()
            )

        else:

            generated_len = int(
                generated_tokens.shape[0]
            )

        generated_lengths.append(
            generated_len
        )

    total_generated_tokens = sum(
        generated_lengths
    )

    average_generated_tokens = (
        total_generated_tokens
        / BATCH_SIZE
    )

    # ========================================================
    # 13. DECODE OUTPUTS
    # ========================================================

    generated_texts = []

    for i in range(BATCH_SIZE):

        generated_tokens = (
            generated_tokens_batch[i]
        )

        # ----------------------------------------------------
        # Remove everything after EOS.
        # ----------------------------------------------------

        eos_positions = (
            generated_tokens
            == tokenizer.eos_token_id
        )

        eos_indices = torch.nonzero(
            eos_positions,
            as_tuple=False
        )

        if eos_indices.numel() > 0:

            eos_index = int(
                eos_indices[0].item()
            )

            generated_tokens = (
                generated_tokens[:eos_index]
            )

        # ----------------------------------------------------
        # Decode only generated tokens.
        # ----------------------------------------------------

        generated_text = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True,
        )

        generated_texts.append(
            generated_text
        )

    # ========================================================
    # 14. OUTPUT
    # ========================================================

    print()
    print("=" * 80)
    print("GENERATED REASONING")
    print("=" * 80)

    for i, generated_text in enumerate(
        generated_texts
    ):

        print()
        print(
            f"--- SAMPLE {i} "
            f"({generated_lengths[i]} tokens) ---"
        )

        print(
            generated_text
        )

    # ========================================================
    # 15. GENERATION STATISTICS
    # ========================================================

    print()
    print("=" * 80)
    print("GENERATION STATISTICS")
    print("=" * 80)

    print(
        f"Batch size               : "
        f"{BATCH_SIZE}"
    )

    print(
        f"Total input tokens       : "
        f"{total_actual_input_tokens}"
    )

    print(
        f"Average input tokens     : "
        f"{total_actual_input_tokens / BATCH_SIZE:.1f}"
    )

    print(
        f"Max padded input length  : "
        f"{padded_input_length}"
    )

    print(
        f"Total generated tokens   : "
        f"{total_generated_tokens}"
    )

    print(
        f"Average generated/sample : "
        f"{average_generated_tokens:.1f}"
    )

    print(
        f"Generation time          : "
        f"{generation_time:.2f} sec"
    )

    if generation_time > 0:

        print(
            f"Generated tokens/sec     : "
            f"{total_generated_tokens / generation_time:.2f}"
        )

        print(
            f"Samples/sec              : "
            f"{BATCH_SIZE / generation_time:.2f}"
        )

    # ========================================================
    # 16. VRAM SUMMARY
    # ========================================================

    print()
    print("=" * 80)
    print("TOTAL VRAM SUMMARY")
    print("=" * 80)

    print(
        f"Initial allocated       : "
        f"{sum(initial_allocated):.3f} GB"
    )

    print(
        f"Peak allocated          : "
        f"{total_peak_allocated:.3f} GB"
    )

    print(
        f"Generation VRAM increase: "
        f"{total_peak_allocated - sum(initial_allocated):.3f} GB"
    )

    print(
        f"Initial reserved        : "
        f"{sum(initial_reserved):.3f} GB"
    )

    print(
        f"Peak reserved           : "
        f"{total_peak_reserved:.3f} GB"
    )

    print()
    print("=" * 80)
    print("PER-GPU PEAK SUMMARY")
    print("=" * 80)

    for i in range(num_gpus):

        print(
            f"GPU {i}: "
            f"{peak_allocated[i]:.3f} GB allocated / "
            f"{peak_reserved[i]:.3f} GB reserved"
        )

    print()
    print(
        "✓ BATCH GENERATION COMPLETED"
    )

BATCH 16 TEACHER GENERATION
LEFT PADDING + GENERATION + VRAM PROFILING

✓ Created 16 teacher prompts

BATCH INPUT
Batch size               : 16
Padding side             : left
Batch tensor shape       : (16, 985)
Padded input length      : 985
Min actual input tokens  : 742
Max actual input tokens  : 985
Average actual tokens    : 853.2
Total actual input tokens: 13652

GPU INFORMATION
Number of GPUs: 2
GPU 0: Tesla T4 (14.562 GB)
GPU 1: Tesla T4 (14.562 GB)

INITIAL VRAM
GPU 0 | Allocated: 3.741 GB | Reserved: 3.762 GB
GPU 1 | Allocated: 3.768 GB | Reserved: 3.789 GB
TOTAL    | Allocated: 7.509 GB | Reserved: 7.551 GB

GENERATING

PEAK VRAM
GPU 0 | Peak allocated: 10.467 GB | Peak reserved: 11.648 GB
GPU 1 | Peak allocated: 10.660 GB | Peak reserved: 12.162 GB

TOTAL | Peak allocated: 21.127 GB
TOTAL | Peak reserved : 23.811 GB

GENERATED REASONING

--- SAMPLE 0 (134 tokens) ---
The question asks for the spouse of the Green performer. To answer this, we first need to identify who the 

In [48]:
# ============================================================
# RESET GPU AFTER GENERATION
# ============================================================

import gc
import torch

print("=" * 80)
print("RESET GPU MEMORY")
print("=" * 80)

# ------------------------------------------------------------
# Delete generation tensors
# ------------------------------------------------------------

for var_name in [
    "outputs",
    "inputs",
]:
    if var_name in globals():
        del globals()[var_name]

# ------------------------------------------------------------
# Python garbage collection
# ------------------------------------------------------------

gc.collect()

# ------------------------------------------------------------
# Clear CUDA cache + IPC
# ------------------------------------------------------------

if torch.cuda.is_available():

    for i in range(torch.cuda.device_count()):

        with torch.cuda.device(i):

            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

            # Reset profiling statistics
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.reset_accumulated_memory_stats()

# ------------------------------------------------------------
# Verify VRAM
# ------------------------------------------------------------

if torch.cuda.is_available():

    print()
    print("=" * 80)
    print("GPU MEMORY AFTER RESET")
    print("=" * 80)

    total_allocated = 0
    total_reserved = 0

    for i in range(torch.cuda.device_count()):

        allocated = (
            torch.cuda.memory_allocated(i)
            / 1024**3
        )

        reserved = (
            torch.cuda.memory_reserved(i)
            / 1024**3
        )

        total_allocated += allocated
        total_reserved += reserved

        print(
            f"GPU {i} | "
            f"Allocated: {allocated:.3f} GB | "
            f"Reserved: {reserved:.3f} GB"
        )

    print(
        f"TOTAL    | "
        f"Allocated: {total_allocated:.3f} GB | "
        f"Reserved: {total_reserved:.3f} GB"
    )

print()
print("✓ Generation tensors deleted")
print("✓ Python garbage collected")
print("✓ CUDA cache cleared")
print("✓ Peak statistics reset")

RESET GPU MEMORY

GPU MEMORY AFTER RESET
GPU 0 | Allocated: 3.741 GB | Reserved: 3.762 GB
GPU 1 | Allocated: 3.768 GB | Reserved: 3.789 GB
TOTAL    | Allocated: 7.508 GB | Reserved: 7.551 GB

✓ Generation tensors deleted
✓ Python garbage collected
✓ CUDA cache cleared
✓ Peak statistics reset
